In [ ]:
##preparing_for_fine_tuning##

# ==========================================
# STEP 1: Import dataset (from Kaggle Hub)
# ==========================================

import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

#!pip install -q kagglehub openpyxl

import kagglehub
import pandas as pd

path = kagglehub.dataset_download("sanak2000/cleveland-clinic-patients-feedback")
print("Dataset folder:", path)
print("Files inside:", os.listdir(path))

file_name = "patient_feedback_dataset.xlsx"
data = pd.read_excel(os.path.join(path, file_name))

print(f"Total rows: {len(data)}")
print("Columns:", data.columns.tolist())
print(data.head())


In [ ]:
# ==========================================
# STEP 2: Clean the text
# ==========================================
import re

# Auto-detect the text column (common names in this dataset)
text_col_candidates = ['review', 'Review', 'feedback', 'Feedback', 'comment', 'Comment', 'text', 'Text']
TEXT_COL = next((c for c in text_col_candidates if c in data.columns), data.select_dtypes(include='object').columns[0])
print(f"Using text column: {TEXT_COL}")

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data['cleaned_text'] = data[TEXT_COL].apply(clean_text)
print(data['cleaned_text'].head())


In [ ]:

# ==========================================
# STEP 3: Handle missing data + build labels
# ==========================================
print(data.isnull().sum())
data['cleaned_text'] = data['cleaned_text'].fillna('unknown')

# Auto-detect rating/label column
rating_candidates = ['rating', 'Rating', 'stars', 'Stars', 'score', 'Score', 'sentiment', 'Sentiment', 'label', 'Label']
LABEL_COL = next((c for c in rating_candidates if c in data.columns), None)
print(f"Using label column: {LABEL_COL}")

def to_sentiment(val):
    # If numeric rating (e.g., 1-5 stars) -> map to negative/neutral/positive
    try:
        v = float(val)
        if v <= 2:   return 2  # negative
        if v == 3:   return 1  # neutral
        return 0               # positive
    except (ValueError, TypeError):
        s = str(val).strip().lower()
        mapping = {"positive": 0, "pos": 0,
                   "neutral": 1, "neu": 1,
                   "negative": 2, "neg": 2}
        return mapping.get(s, None)

data['label'] = data[LABEL_COL].apply(to_sentiment)
data = data.dropna(subset=['label']).reset_index(drop=True)
data['label'] = data['label'].astype(int)

print(data['label'].value_counts())



In [ ]:

# ==========================================
# STEP 4: Tokenization
# ==========================================
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

tokens = tokenizer(
    data['cleaned_text'].tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

print(tokens['input_ids'][:5])

# ---- Synonym replacement (augmentation) ----
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.corpus import wordnet

def synonym_replacement(word):
    synonyms = wordnet.synsets(word)
    if synonyms:
        return random.choice(synonyms).lemmas()[0].name()
    return word

def augment_text(text):
    words = text.split()
    augmented_words = [
        synonym_replacement(w) if random.random() > 0.8 else w
        for w in words
    ]
    return ' '.join(augmented_words)

data['augmented_text'] = data['cleaned_text'].apply(augment_text)



In [ ]:

# ==========================================
# STEP 5: Structure the data for fine-tuning
# ==========================================
import torch
from torch.utils.data import TensorDataset, DataLoader

input_ids = tokens['input_ids']
attention_masks = tokens['attention_mask']
labels = torch.tensor(data['label'].tolist(), dtype=torch.long)



In [ ]:

# ==========================================
# STEP 6: Split the Dataset
# ==========================================
from sklearn.model_selection import train_test_split

train_val_inputs, test_inputs, train_val_masks, test_masks, train_val_labels, test_labels = train_test_split(
    input_ids, attention_masks, labels, test_size=0.15, random_state=42
)

train_inputs, val_inputs, train_masks, val_masks, train_labels, val_labels = train_test_split(
    train_val_inputs, train_val_masks, train_val_labels, test_size=0.2, random_state=42
)

train_dataset = TensorDataset(train_inputs, train_masks, train_labels)
val_dataset   = TensorDataset(val_inputs,   val_masks,   val_labels)
test_dataset  = TensorDataset(test_inputs,  test_masks,  test_labels)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=16)
test_dataloader  = DataLoader(test_dataset,  batch_size=16)

print("Training, validation, and test sets are prepared with attention masks!")


In [ ]:

# ==========================================
# STEP 7: Configure & Fine-tune
# ==========================================
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

def to_hf_dataset(inputs, masks, lbls):
    return Dataset.from_dict({
        "input_ids":      inputs.tolist(),
        "attention_mask": masks.tolist(),
        "label":          lbls.tolist(),
    })

hf_train = to_hf_dataset(train_inputs, train_masks, train_labels)
hf_val   = to_hf_dataset(val_inputs,   val_masks,   val_labels)
hf_test  = to_hf_dataset(test_inputs,  test_masks,  test_labels)

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=3
)

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    logging_dir='./logs',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
)

print("Starting Fine-tuning on the Cleveland Clinic feedback dataset...")
trainer.train()



In [ ]:

# ==========================================
# STEP 8: Evaluate
# ==========================================
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

predictions_output = trainer.predict(hf_test)
preds = predictions_output.predictions.argmax(-1)
true_labels = hf_test["label"]

print(f"Test Accuracy:  {accuracy_score(true_labels, preds):.4f}")
print(f"Test F1 Score:  {f1_score(true_labels, preds, average='weighted'):.4f}")
print(f"Test Precision: {precision_score(true_labels, preds, average='weighted'):.4f}")
print(f"Test Recall:    {recall_score(true_labels, preds, average='weighted'):.4f}")


In [ ]:
# ==========================================
# STEP 9: Save the fine-tuned model
# ==========================================
model.save_pretrained("./fine_tuned_bert")
tokenizer.save_pretrained("./fine_tuned_bert")
print("\nSuccess! Your model trained on the Cleveland Clinic dataset is saved in './fine_tuned_bert'.")


In [ ]:
pip list --format=freeze > requirements.txt